# Session 5 — Student Exercise
## TenderScope Europe: build a weekly opportunity monitor

**Individual exercise — static web scraping**

## Business objective

A Data & AI consulting practice wants a repeatable weekly shortlist of public tender opportunities.

An opportunity enters the final shortlist when all four conditions are true:

1. the notice status is **Open**;
2. at least one published topic belongs to the practice's capability list;
3. the estimated budget is at least **EUR 200,000**;
4. the submission deadline is on or after **21 October 2026**.

The shortlist is a prioritisation aid. It is not an automatic bid decision.

## Technical setup

Because the simulated website is stored locally, the next cells expose it through a temporary local HTTP URL. On a real website, `FIRST_PAGE_URL` would be the public URL.

In [1]:
from datetime import datetime, timezone
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from threading import Thread
from urllib.parse import urljoin
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, display

In [2]:
def find_exercise_site(starting_path):
    current_path = starting_path.resolve()

    for folder in [current_path, *current_path.parents]:
        candidates = [
            folder / "data" / "S5_HTML_SITES" / "02_student_exercise_tenderscope",
            folder / "S5_HTML_SITES" / "02_student_exercise_tenderscope",
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate.resolve()

    raise FileNotFoundError(
        "TenderScope site not found. Expected: "
        "Session5/data/S5_HTML_SITES/02_student_exercise_tenderscope"
    )


SITE_ROOT = find_exercise_site(Path.cwd())
print("Exercise site:", SITE_ROOT)

Exercise site: /Users/guillaumeleorat/Projects/Teaching/AlbertSchool/Advanced_EDA_DataCollection/Students/Session5/data/S5_HTML_SITES/02_student_exercise_tenderscope


In [3]:
# Restart the temporary local server safely if this cell is executed again.
if "server" in globals():
    try:
        server.shutdown()
        server.server_close()
    except Exception:
        pass


class QuietHandler(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
        ".txt": "text/plain; charset=utf-8",
    }

    def log_message(self, format, *args):
        pass


handler = partial(QuietHandler, directory=str(SITE_ROOT))
server = ThreadingHTTPServer(("127.0.0.1", 0), handler)
Thread(target=server.serve_forever, daemon=True).start()

BASE_URL = f"http://127.0.0.1:{server.server_port}"
FIRST_PAGE_URL = f"{BASE_URL}/index.html"

print("Local site ready:", FIRST_PAGE_URL)

Local site ready: http://127.0.0.1:64258/index.html


## 1. Inspect the website before writing extraction code

Open the page and investigate it as an analyst:

- Is the target information visible on the results page?
- Is it also present in **View Page Source**?
- How many result pages exist?
- Does the URL change when moving between pages?
- Which information is available only on a detail page?
- Are all visually similar cards genuine tender notices?
- Can the same notice appear more than once?

Open the local governance files directly in VS Code:

- `data/S5_HTML_SITES/02_student_exercise_tenderscope/robots.txt`
- `data/S5_HTML_SITES/02_student_exercise_tenderscope/terms.html`

In [4]:
display(HTML(
    f'<a href="{FIRST_PAGE_URL}" target="_blank" '
    'style="font-size:18px;font-weight:700">Open TenderScope Europe ↗</a>'
))

### Analyst checkpoint

Write down your decisions before coding:

- **Unit of analysis:** ...
- **Fields needed for the business rule:** ...
- **Evidence that the page is static for this target:** ...
- **Access/reuse decision and evidence:** ...
- **Main collection risks:** ...

## 2. Request the first page and confirm the target is in the HTML

In [ ]:
headers = {"User-Agent": "AlbertSchool-S5-StudentCollector/1.0"}

# TODO — request FIRST_PAGE_URL, raise an error for a bad HTTP response,
# then print the status code and the number of characters received.

In [ ]:
known_title = "National AI Procurement Observatory"

# TODO — verify that known_title exists in the HTML returned by requests.
# This is the key static-page test for our target information.

## 3. Discover the relevant HTML structure

Do not begin with a selector supplied by someone else. Begin with a title you can see in the browser, locate it in the parsed HTML, and inspect its nearest record container.

In [ ]:
# TODO — parse response.text with BeautifulSoup.
soup = None

# TODO — find the text node containing known_title.
title_node = None

# TODO — move to the nearest article or section containing the complete notice.
candidate_record = None

# Display the structure and the attributes you discovered.
# print(candidate_record.prettify())
# print(candidate_record.attrs)

### Selector decision

Answer before continuing:

- Which tag contains one complete tender listing?
- Which attribute distinguishes a tender from an event or a hidden template?
- Why would selecting every `.result-card` be unsafe?
- What selector will you use, and why is it defensible?

In [ ]:
# TODO — write a semantic CSS selector based on the structure you inspected.
TENDER_SELECTOR = ""

# TODO — select the genuine tender cards on page 1 and inspect the count.
tender_cards = []
print("Tender cards found on page 1:", len(tender_cards))

## 4. Extract one listing first

Start with one record. Preserve both the visible fields and the link to the detail page.

In [ ]:
# TODO — extract one tender as a dictionary with these fields:
# tender_id, title, buyer, country, published_text, deadline_text,
# status, topics, detail_url.
one_listing = {}

one_listing

## 5. Create a reusable listing parser

In [ ]:
def parse_listing_card(card, page_url):
    # TODO — convert one result card into one dictionary.
    # Use urljoin(page_url, relative_link) for the detail URL.
    raise NotImplementedError("Complete parse_listing_card")

In [ ]:
# TODO — test the function on every genuine tender card from page 1.
page_1_records = []
page_1_df = pd.DataFrame(page_1_records)

display(page_1_df)

## 6. Discover and collect all result pages

Do not hard-code four unrelated URLs. Use the pagination links published by the first page.

In [ ]:
# TODO — build a unique list of absolute result-page URLs from nav.pagination.
page_urls = []

for url in page_urls:
    print(url)

In [ ]:
def collect_result_page(url):
    # TODO — request one page, parse it, select genuine tender cards,
    # convert them to records and attach the result-page URL.
    raise NotImplementedError("Complete collect_result_page")

In [ ]:
# TODO — collect every result page and combine the records.
# Respect the site's instruction of one request per second.
listing_records = []

listings_raw = pd.DataFrame(listing_records)
display(listings_raw.head())

## 7. Validate the listings and resolve duplicates

The portal reports **30 listings**, but a listing is not necessarily a unique business opportunity.

Decide:

- What is the stable identifier?
- Which rows are duplicated?
- At what stage should duplicates be removed?
- Which duplicate should be retained, and why?

In [ ]:
# TODO — inspect counts, missing identifiers and duplicated tender IDs.

In [ ]:
# TODO — create one row per unique tender notice.
listings = pd.DataFrame()

print("Unique notices:", len(listings))

## 8. Visit each unique detail page

The estimated budget is not present on the result cards. It is published on each tender's detail page.

Request detail pages only after deduplication so that repeated listings do not create repeated requests.

In [ ]:
def parse_detail_page(url):
    # TODO — request and parse one detail page.
    # Return: detail_tender_id, budget_text, procedure and detail_source_url.
    raise NotImplementedError("Complete parse_detail_page")

In [ ]:
# TODO — collect every unique detail page politely.
detail_records = []

details = pd.DataFrame(detail_records)
display(details.head())

In [ ]:
# TODO — merge listings and details by tender ID.
# Use merge validation to make the expected relationship explicit.
tenders_raw = pd.DataFrame()

display(tenders_raw.head())

## 9. Convert dates and budgets without losing the source text

Preserve `deadline_text` and `budget_text` as evidence. Add separate analytical fields:

- `deadline`
- `budget_eur`

The source contains several date and budget formats, including `k`, `million`, and `Not disclosed`.

In [ ]:
def parse_budget_eur(text):
    # TODO — return a numeric EUR value or pd.NA when the budget is not disclosed.
    raise NotImplementedError("Complete parse_budget_eur")

In [ ]:
# TODO — create tenders from tenders_raw.
# Parse mixed dates and apply parse_budget_eur to budget_text.
tenders = pd.DataFrame()

display(tenders[["tender_id", "deadline_text", "deadline", "budget_text", "budget_eur"]].head())

## 10. Validate the canonical dataset

Create explicit checks for at least:

- four result pages;
- 30 collected listings;
- 28 unique notices;
- unique tender IDs after deduplication;
- one matching detail page per notice;
- known statuses;
- parseable deadlines;
- retained source URLs.

Add any other checks you consider important.

In [ ]:
# TODO — implement and display the validation checks.
checks = pd.Series(dtype="bool")
display(checks.to_frame("passed"))

## 11. Apply the business rules

Capability topics:

```python
CAPABILITY_TOPICS = {
    "Artificial Intelligence", "Customer Data", "Analytics",
    "Machine Learning", "Business Intelligence", "Generative AI",
    "Data Platform", "Data Governance", "Geospatial", "Open Data",
    "MLOps", "Forecasting", "Data Quality", "Data Integration",
    "AI Ethics", "Master Data"
}
```

Explain your treatment of notices whose budget is `Not disclosed`.

In [ ]:
CAPABILITY_TOPICS = {
    "Artificial Intelligence", "Customer Data", "Analytics",
    "Machine Learning", "Business Intelligence", "Generative AI",
    "Data Platform", "Data Governance", "Geospatial", "Open Data",
    "MLOps", "Forecasting", "Data Quality", "Data Integration",
    "AI Ethics", "Master Data",
}

# TODO — create Boolean fields for each business condition.
# TODO — produce a final shortlist sorted by descending budget.
shortlist = pd.DataFrame()

display(shortlist)

## 12. Business handover

Provide a concise conclusion:

- number of opportunities on the shortlist;
- top three by estimated budget;
- notices excluded because the budget is not disclosed;
- two checks a business developer should still perform before deciding to bid;
- one structural change that could break your collector next week.

### Your conclusion

...